# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [3]:
# Combine all pages into a single string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Print the number of pages loaded and the first part of the document
print(f"Number of pages loaded: {len(docs)}")
print(document_text[:1000])  # Preview the first 1000 characters of the document

Number of pages loaded: 26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies 

In [4]:
docs[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}, page_content='pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025')

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI() 

#Define output schema
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


In [6]:
#instructions (Developer) prompt
instructions = """
You are an expert assistant for AI professionals. 
Generate a structured output as a Pydantic BaseModel with the following fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens. 
Use the specified tone. Your summary should be succinct, max 1000 tokens.
"""

#set the output tone and user prompt
article_context = document_text
tone = "Formal Academic Writing"

user_prompt = f"""
Please create a structured output from the pdf using  the following context:
<Context>
{article_context}
</Context>

Produce a summary using the following tone:
<tone>
{tone}
</tone>

Provide your response in the following format:
- Author: <author>
- Title: <title>
- Relevance: a statement, no longer than one paragraph, 
that explains why is this article relevant for an AI professional in their professional development.
- Summary: concise, max 1000 tokens <summary>
- Tone
- InputTokens: <number of input tokens>
- OutputTokens: <number of tokens in the output>

Output should be a Pydantic BaseModel object.
"""

In [7]:
response = client.responses.parse(
    model="gpt-4o",
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
        ],
    temperature=1.2,
    text_format=ArticleSummary,
)


In [8]:
event = response.output_parsed

event

ArticleSummary(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='The article is crucial for AI professionals to understand the challenges and strategies of AI adoption and scaling within enterprises. It provides insights into successful practices for integrating AI systems that continuously learn and adapt, making it relevant for shaping AI strategies in business contexts.', Summary="The report emphasizes the growing 'GenAI Divide,' highlighting that only 5% of AI pilots result in significant business transformations, despite widespread adoption. It identifies learning capability as a core barrier, illustrating that many AI tools fail to integrate and adapt to workflows, leading to a high implementation failure rate. Organizations find more success with external partnerships that deliver customized, learning-capable systems. Meanwhile, a 'shadow AI economy' has emerged, where employees use consum

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase


In [10]:
input_text = document_text  #the original, factual content
summary_text = event.Summary  #summary produced in the previous step 


In [11]:
test_case = LLMTestCase(input=input_text, actual_output=summary_text)
metric = SummarizationMetric(
    threshold=0.7,
    model="gpt-4o",
    assessment_questions=[
    "Does the summary accurately reflect the main ideas and arguments of the source?",
    "Are all critical facts and findings present in the summary?",
    "Is there any information in the summary that is not supported by the source document?",
    "Does the summary provide a coherent understanding of the document's purpose?",
    "Is the summary suitable for use by an AI professional seeking to update their knowledge?"
    ],
    include_reason=True
)

metric.measure(test_case)
print(metric.score)
print(metric.reason)

Output()

0.5
The score is 0.50 because the summary includes extra information not present in the original text, such as specific sectors experiencing GenAI impacts, which were not detailed in the source. Additionally, the summary fails to address key questions about accurately reflecting the main ideas and ensuring all critical facts are present, indicating a lack of comprehensive coverage.


In [12]:
coherence_metric = GEval(
    name = "Coherence",
    threshold=0.7,
    model="gpt-4o",
    evaluation_steps=[
        "Check if the response uses clear and direct language.",
        "Determine if ideas progress logically and smoothly.",
        "Verify the explanation avoids jargon or defines it when used.",
        "Assess whether complex ideas are presented clearly.",
        "Identify any vague or confusing parts."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
coherence_metric.measure(test_case)
print(coherence_metric.score)
print(coherence_metric.reason)

Output()

0.8106192110078881
The response uses clear and direct language and presents ideas logically, aligning well with the evaluation steps. It effectively summarizes the key findings of the report, such as the GenAI Divide and the importance of learning-capable systems, without using jargon. However, it could improve by providing more specific examples or details from the test case to enhance clarity and avoid any potential vagueness.


In [13]:
tonality_metric = GEval(
    name = "Tonality",
    threshold=0.7,
    model="gpt-4o",
    evaluation_steps=[
        "Check if the tone remains consistent throughout the summary.",
        "Assess whether tone and style fit the specified requirements.",
        "Determine if the language demonstrates professionalism.",
        "Make sure personal opinions are not expressed.",
        "Judge whether the tone is suitable for AI professionals."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
tonality_metric.measure(test_case)
print(tonality_metric.score)
print(tonality_metric.reason)

Output()

0.8247524429446955
The response maintains a consistent tone and style suitable for AI professionals, demonstrating professionalism throughout. It avoids personal opinions and aligns well with the specified requirements by focusing on the key findings and implications of the report. However, it could have been more explicit in detailing how the tone fits the specified requirements, and it slightly lacks in-depth analysis of the tone's suitability for AI professionals.


In [14]:
safety_metric = GEval(
    name = "Tonality",
    threshold=0.7,
    model="gpt-4o",
    evaluation_steps=[
        "Check if the tone remains consistent throughout the summary.",
        "Assess whether tone and style fit the specified requirements.",
        "Determine if the language demonstrates professionalism.",
        "Make sure personal opinions are not expressed.",
        "Judge whether the tone is suitable for AI professionals."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
safety_metric.measure(test_case)
print(safety_metric.score)
print(safety_metric.reason)

Output()

0.8509546094596757
The response maintains a consistent and professional tone suitable for AI professionals, aligning well with the evaluation steps. It avoids personal opinions and focuses on summarizing the report's findings, such as the 'GenAI Divide' and the importance of learning-capable systems. The response effectively highlights key details like the success of external partnerships and the emergence of a 'shadow AI economy,' demonstrating a clear understanding of the document's content. However, it could slightly improve by explicitly mentioning the tone and style requirements.


In [15]:
# Structured result output
results = {
    "SummarizationScore": metric.score,
    "SummarizationReason": metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

import json
print(json.dumps(results, indent=2))

{
  "SummarizationScore": 0.5,
  "SummarizationReason": "The score is 0.50 because the summary includes extra information not present in the original text, such as specific sectors experiencing GenAI impacts, which were not detailed in the source. Additionally, the summary fails to address key questions about accurately reflecting the main ideas and ensuring all critical facts are present, indicating a lack of comprehensive coverage.",
  "CoherenceScore": 0.8106192110078881,
  "CoherenceReason": "The response uses clear and direct language and presents ideas logically, aligning well with the evaluation steps. It effectively summarizes the key findings of the report, such as the GenAI Divide and the importance of learning-capable systems, without using jargon. However, it could improve by providing more specific examples or details from the test case to enhance clarity and avoid any potential vagueness.",
  "TonalityScore": 0.8247524429446955,
  "TonalityReason": "The response maintains

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [16]:
#combine evaluation feedback
feedback = (
    f"Summarization evaluation: {results['SummarizationReason']}\n\n"
    f"Coherence evaluation: {results['CoherenceReason']}\n\n"
    f"Tonality evaluation: {results['TonalityReason']}\n\n"
    f"Safety evaluation: {results['SafetyReason']}\n"
)
feedback

"Summarization evaluation: The score is 0.50 because the summary includes extra information not present in the original text, such as specific sectors experiencing GenAI impacts, which were not detailed in the source. Additionally, the summary fails to address key questions about accurately reflecting the main ideas and ensuring all critical facts are present, indicating a lack of comprehensive coverage.\n\nCoherence evaluation: The response uses clear and direct language and presents ideas logically, aligning well with the evaluation steps. It effectively summarizes the key findings of the report, such as the GenAI Divide and the importance of learning-capable systems, without using jargon. However, it could improve by providing more specific examples or details from the test case to enhance clarity and avoid any potential vagueness.\n\nTonality evaluation: The response maintains a consistent tone and style suitable for AI professionals, demonstrating professionalism throughout. It av

In [17]:
enhancement_instruction = (
    "You are tasked to revise and enhance the summary below using context from the original article. "
    "Address the points mentioned in the evaluation feedback to improve factual coverage, coherence, tone, and safety for AI professionals. "
    "Your improved summary should be concise (max 1000 tokens), formal, and relevant."
)

In [18]:
enhanced_prompt = (f"""
    <Context>
    {document_text}
    </Context>

    <PreviousSummary>
    {summary_text}
    </PreviousSummary>
    
    <EvaluationFeedback>
    {feedback}</EvaluationFeedback>

    Please rewrite and enhance the summary based on the evaluation, correcting any identified issues. 
    Use a professional and clear tone.
    """
)

In [19]:
#generate improved summary using OpenAI
new_response = client.responses.parse(
    model="gpt-4o",
    input=[
        {"role": "system", "content": enhancement_instruction},
        {"role": "user", "content": enhanced_prompt},
        ],
    temperature=1.2,
    text_format=ArticleSummary,
)


In [20]:
new_event = new_response.output_parsed

new_event

ArticleSummary(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025 - MIT NANDA', Relevance='AI Implementation and Strategy', Summary="The report by MIT NANDA explores the 'GenAI Divide,' revealing that despite $30-40 billion in enterprise investments in generative AI, only 5% result in significant business transformations. The primary barrier is identified as the lack of learning capabilities. Most AI tools struggle to integrate effectively into workflows, leading to a high failure rate. Successful organizations turn to external partnerships that provide customized, learning-capable systems instead of relying on internal builds. Meanwhile, a 'shadow AI economy' has developed as employees independently utilize consumer AI tools.\n\nIndustry disruptions are mostly confined to the Technology and Media sectors, with seven out of nine sectors not experiencing substantial change. Organizations that proceed beyond pil

In [21]:
#new summarised metric results
test_case_enhanced = LLMTestCase(input=input_text, actual_output=new_event.Summary)
new_summetric = SummarizationMetric(
    threshold=0.7,
    model="gpt-4o",
    assessment_questions=[
    "Does the summary accurately reflect the main ideas and arguments of the source?",
    "Are all critical facts and findings present in the summary?",
    "Is there any information in the summary that is not supported by the source document?",
    "Does the summary provide a coherent understanding of the document's purpose?",
    "Is the summary suitable for use by an AI professional seeking to update their knowledge?"
    ],
    include_reason=True
)

new_summetric.measure(test_case_enhanced)
print(new_summetric.score)
print(new_summetric.reason)

Output()

0.6666666666666666
The score is 0.67 because the summary contains a significant contradiction regarding the percentage of investments leading to business transformations, which misrepresents the original text's data. Additionally, the summary introduces extra information about AI integration challenges and sector-specific disruptions that are not present in the original text. These discrepancies and omissions affect the overall accuracy and completeness of the summary, justifying the given score.


In [22]:
#compare and analyse
better = new_summetric.score > metric.score
if better:
    print("The enhanced summary performed better in evaluation. This self-correction loop with feedback and auto-evaluation improved the output.")
else:
    print("The enhanced summary did not outperform the initial version. This may be due to limitations in model understanding or insufficient feedback detail.")

The enhanced summary performed better in evaluation. This self-correction loop with feedback and auto-evaluation improved the output.


In [23]:
new_coherence_metric = GEval(
    name = "Coherence",
    threshold=0.7,
    model="gpt-4o",
    evaluation_steps=[
        "Check if the response uses clear and direct language.",
        "Determine if ideas progress logically and smoothly.",
        "Verify the explanation avoids jargon or defines it when used.",
        "Assess whether complex ideas are presented clearly.",
        "Identify any vague or confusing parts."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
new_coherence_metric.measure(test_case_enhanced)
print(new_coherence_metric.score)
print(new_coherence_metric.reason)

Output()

0.842490791433043
The response uses clear and direct language and presents ideas logically, aligning well with the evaluation steps. It effectively summarizes the key findings of the report, such as the GenAI Divide and the importance of learning-capable systems, without using jargon. However, it could improve by providing more specific examples from the test case, such as the role of 'shadow AI economy' or the specific sectors affected by the GenAI Divide, to enhance clarity and avoid any potential vagueness.


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
